SISTEMA DE RECOMENDACIÓN DE CURSOS CON FILTRO DE PRESUPUESTO

Contexto: Una plataforma de EdTech de cursos en línea desea implementar un motor de recomendación basado en contenido. El sistema debe sugerir cursos similares en contenido textual a uno dado, pero con una condición de negocio estricta: el sistema solo debe recomendar cursos cuyo precio sea menor o igual al presupuesto máximo definido por el usuario.

Dataset Inicial de Entrada: Usted cuenta con un DataFrame (df_cursos) que contiene las columnas:

CursoID: Identificador único del curso.

Titulo: Nombre del curso.

Descripcion: Resumen temático del contenido del curso (texto limpio).

Precio: Costo actual del curso en dólares (numérico).

Se le solicita desarrollar:

1. Similitud de Coseno: Calcule la matriz de similitud de coseno entre los vectores de los cursos basados en la vectorización TF-IDF de sus descripciones.

2. Función de Recomendación: Cree una función obtener_similares_por_contenido(curso_id, top_n=3) que devuelva los títulos de los cursos más parecidos al ID ingresado.

3. Integración: Modifique la función para que reciba un parámetro adicional llamado precio_max. El sistema deberá evaluar en tiempo real los costos y recomendar únicamente los cursos que no superen dicho presupuesto.

In [5]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [3]:
data = {
    'CursoID': ['C01', 'C02', 'C03', 'C04', 'C05'],
    'Titulo': [
        'Fundamentos de Data Engineering y SQL',
        'Arquitectura de Datos Avanzada en Cloud y Databricks',
        'Principios de Finanzas y Contabilidad Básica',
        'Machine Learning y Pipelines de Datos Eficientes',
        'Ciberseguridad Esencial y Redes Seguras'
    ],
    'Descripcion': [
        'introduccion al almacenamiento procesamiento de datos pipelines ingenieria de datos masivos y sql',
        'arquitectura avanzada cloud ingenieria de datos lagos de datos databricks y optimizacion de pipelines',
        'conceptos financieros basicos balances contables administracion de dinero e inversiones corporativas',
        'modelos avanzados de machine learning pipelines de datos ciencia de datos e inteligencia artificial',
        'seguridad de la informacion proteccion de redes infraestructura segura y mitigacion de ataques informaticos'
    ],
    'Precio': [50.0, 120.0, 30.0, 95.0, 45.0]  # Variable numérica para el filtro
}

df_cursos = pd.DataFrame(data)

In [4]:
df_cursos

,CursoID,Titulo,Descripcion,Precio
0,C01,Fundamentos de Data Engineering y SQL,introduccion al almacenamiento procesamiento d...,50.0
1,C02,Arquitectura de Datos Avanzada en Cloud y Data...,arquitectura avanzada cloud ingenieria de dato...,120.0
2,C03,Principios de Finanzas y Contabilidad Básica,conceptos financieros basicos balances contabl...,30.0
3,C04,Machine Learning y Pipelines de Datos Eficientes,modelos avanzados de machine learning pipeline...,95.0
4,C05,Ciberseguridad Esencial y Redes Seguras,seguridad de la informacion proteccion de rede...,45.0


In [6]:
# Vectorización TF-IDF de las descripciones
tfidf = TfidfVectorizer()
X_tfidf = tfidf.fit_transform(df_cursos['Descripcion'])

4.1 Similitud de Coseno

In [7]:
# Calculamos la matriz de similitud de coseno a partir de las representaciones vectoriales TF-IDF
matriz_similitud = cosine_similarity(X_tfidf, X_tfidf)

4.2 Recomendar cursos

In [8]:
def obtener_similares_por_contenido(id_referencia, cantidad=3):
    # Validar existencia del ID de manera alternativa
    if id_referencia not in df_cursos['CursoID'].values:
        return f"El identificador {id_referencia} no existe en los registros."

    # Obtener el índice numérico del curso de consulta
    idx_origen = df_cursos[df_cursos['CursoID'] == id_referencia].index[0]

    # Extraer la fila de similitudes de la matriz
    fila_similitudes = matriz_similitud[idx_origen]

    # np.argsort devuelve los índices ordenados de MENOR a MAYOR similitud.
    # Con [::-1] invertimos el arreglo para tener los más similares al principio.
    indices_ordenados = np.argsort(fila_similitudes)[::-1]

    titulos_sugeridos = []

    for idx in indices_ordenados:
        # Excluir el curso de origen
        if idx == idx_origen:
            continue

        # Extraer el título directamente por posición indexada de Pandas
        titulos_sugeridos.append(df_cursos.at[idx, 'Titulo'])

        # Validar el límite solicitado
        if len(titulos_sugeridos) == cantidad:
            break

    return titulos_sugeridos

In [10]:
id_test = 'C01'
print("Curso consultado:", df_cursos[df_cursos['CursoID'] == id_test]['Titulo'].values[0])
print("-" * 75)
print("Nueva función 'obtener_similares_por_contenido':")
print(obtener_similares_por_contenido(id_referencia=id_test, cantidad=2))

Curso consultado: Fundamentos de Data Engineering y SQL
---------------------------------------------------------------------------
Nueva función 'obtener_similares_por_contenido':
['Arquitectura de Datos Avanzada en Cloud y Databricks', 'Machine Learning y Pipelines de Datos Eficientes']
